# Completeness measures

ISO/IEC 25024 completeness and ISO/IEC 5259-2 `Com-*` measures: how much of the data that should be there, is there. Note that of these measures need `fit`, because completeness has nothing to learn from clean data, only nulls to count.

In [1]:
import polars as pl

from dqmeasure import (
    EmptyRecords,
    FeatureCompleteness,
    LabelCompleteness,
    RecordCompleteness,
    ValueCompleteness,
    ValueOccurrenceCompleteness,
)

## FeatureCompleteness
**"Is the cell non-null?"**

Column-scoped, cell-level.

In [2]:
readings = pl.DataFrame({"temperature": [21.0, None, 19.5, None, 22.1]})

measure = FeatureCompleteness("temperature")
measure.predict(readings)

temperature
f64
1.0
0.0
1.0
0.0
1.0


In [3]:
measure.score(readings)

0.6

Two of five cells are null.

## LabelCompleteness
**"Is the sample's label present?"**

Numerically the same computation as `FeatureCompleteness`, applied to a label column.

In [4]:
samples = pl.DataFrame({"label": ["cat", "dog", None, "cat", "dog"]})

measure = LabelCompleteness("label")
measure.score(samples)

0.8

One of five samples is missing its label.

## RecordCompleteness
**"Does the record have zero empty cells?"**

Table-scoped: the unit is the whole row, not a single column.

In [5]:
orders = pl.DataFrame(
    {
        "order_id": [1, 2, 3, 4],
        "customer": ["Ana", "Bo", None, "Dee"],
        "total": [42.0, None, 15.0, 8.0],
    }
)

measure = RecordCompleteness()
measure.predict(orders)

record
f64
1.0
0.0
0.0
1.0


In [6]:
measure.score(orders)

0.5

Rows 2 and 3 each have one null cell.

## ValueCompleteness
**"What fraction of all cells in the table are non-null?"**

Same table as above, but `predict` reports each record's own fraction of non-null cells (`Com-I-1`, record completeness) rather than a 0/1 verdict.

In [7]:
measure = ValueCompleteness()
measure.predict(orders)

record
f64
1.0
0.666667
0.666667
1.0


In [8]:
measure.score(orders)

0.8333333333333334

Two of twelve cells are null.

## EmptyRecords
**"Does the record carry any data at all?"**

Table-scoped. Distinct from `RecordCompleteness`: a record with *some* nulls still passes here, it only fails when every cell is empty.

In [9]:
logs = pl.DataFrame({"user": ["ana", None, "bo", None], "action": ["login", None, "logout", "click"]})

measure = EmptyRecords()
measure.predict(logs)

record
f64
1.0
0.0
1.0
1.0


In [10]:
measure.score(logs)

0.75

## ValueOccurrenceCompleteness
**"Does each domain value occur as often as expected?"**

Non-positional, `score`-only: `fit` learns each value's expected share from clean data, and over-represented values can't compensate for missing ones.

In [2]:
clean = pl.DataFrame({"grade": ["A", "A", "B", "B", "C"] * 4})  # 40% A, 40% B, 20% C
skewed = pl.DataFrame({"grade": ["A"] * 15 + ["B"] * 3 + ["C"] * 2})  # A over-represented, B under

measure = ValueOccurrenceCompleteness("grade").fit(clean)
measure.expected_

{'C': 0.2, 'A': 0.4, 'B': 0.4}

In [3]:
measure.score(skewed)

0.65